In [ ]:
import pandas as pd
data = pd.read_csv("/content/boston_housing_data.csv")
print(data)

        CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD  TAX  \
0    0.00632  18.0   2.31   0.0  0.538  6.575  65.2  4.0900    1  296   
1    0.02731   0.0   7.07   0.0  0.469  6.421  78.9  4.9671    2  242   
2    0.02729   0.0   7.07   0.0  0.469  7.185  61.1  4.9671    2  242   
3    0.03237   0.0   2.18   0.0  0.458  6.998  45.8  6.0622    3  222   
4    0.06905   0.0   2.18   0.0  0.458  7.147  54.2  6.0622    3  222   
..       ...   ...    ...   ...    ...    ...   ...     ...  ...  ...   
501  0.06263   0.0  11.93   0.0  0.573  6.593  69.1  2.4786    1  273   
502  0.04527   0.0  11.93   0.0  0.573  6.120  76.7  2.2875    1  273   
503  0.06076   0.0  11.93   0.0  0.573  6.976  91.0  2.1675    1  273   
504  0.10959   0.0  11.93   0.0  0.573  6.794  89.3  2.3889    1  273   
505  0.04741   0.0  11.93   0.0  0.573  6.030   NaN  2.5050    1  273   

     PTRATIO       B  LSTAT  MEDV  
0       15.3  396.90   4.98  24.0  
1       17.8  396.90   9.14  21.6  
2       17.8  3

In [ ]:
data.isnull().sum()

,0
CRIM,20
ZN,20
INDUS,20
CHAS,20
NOX,0
RM,0
AGE,20
DIS,0
RAD,0
TAX,0


In [ ]:
(data.isnull().sum()/len(data))*100

,0
CRIM,3.952569
ZN,3.952569
INDUS,3.952569
CHAS,3.952569
NOX,0.000000
RM,0.000000
AGE,3.952569
DIS,0.000000
RAD,0.000000
TAX,0.000000


In [ ]:
missing_values = data[['CRIM','ZN','INDUS','CHAS','AGE','LSTAT']]
data[missing_values.columns] = missing_values.fillna(missing_values.median())

In [ ]:
data.isnull().sum()

,0
CRIM,0
ZN,0
INDUS,0
CHAS,0
NOX,0
RM,0
AGE,0
DIS,0
RAD,0
TAX,0


In [ ]:
def count_outliers(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3.5 * IQR
    upper_bound = Q3 + 3.5 * IQR
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    return len(outliers)

for col in data.columns:
    print(f"{col}: {count_outliers(data, col)} outliers")

CRIM: 40 outliers
ZN: 126 outliers
INDUS: 0 outliers
CHAS: 34 outliers
NOX: 0 outliers
RM: 0 outliers
AGE: 0 outliers
DIS: 0 outliers
RAD: 0 outliers
TAX: 0 outliers
PTRATIO: 0 outliers
B: 54 outliers
LSTAT: 0 outliers
MEDV: 0 outliers


In [ ]:
data.drop(columns='B',inplace=True)

In [ ]:
print(data.shape)

(506, 13)


In [ ]:
for col in ['ZN', 'CRIM']:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3.5 * IQR
    upper_bound = Q3 + 3.5 * IQR
    data = data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]

In [ ]:
print(data.shape)

(368, 13)


In [ ]:
skewness = data.skew(numeric_only=True).sort_values(ascending=False)
print(skewness)

CHAS       3.285840
CRIM       1.925435
MEDV       1.541539
DIS        1.281185
LSTAT      0.802803
NOX        0.738171
RAD        0.687254
RM         0.416619
TAX        0.347653
INDUS      0.034985
ZN         0.000000
PTRATIO   -1.006570
AGE       -1.144760
dtype: float64


In [ ]:
import numpy as np

cols_to_transform = ['CRIM', 'DIS']
for col in cols_to_transform:
    data[col] = np.log1p(data[col])
print(data[cols_to_transform].skew())

CRIM    0.838888
DIS     0.455396
dtype: float64


In [ ]:
X = data[['AGE','CHAS','CRIM','DIS','INDUS','LSTAT','NOX','PTRATIO','RAD','RM','TAX','ZN']]
y = data['MEDV']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
X_train_scaled = scale.fit_transform(X_train)
X_test_scaled = scale.transform(X_test) #only transform to avoid data leak

In [ ]:
from sklearn.linear_model import LinearRegression
linear = LinearRegression()
linear.fit(X_train_scaled, y_train)

y_pred_lnr = linear.predict(X_test_scaled)

In [ ]:
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)

In [ ]:
from sklearn.linear_model import Lasso
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

def evaluate(y_test, y_pred, model_name):
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    print(f"{model_name} -> RMSE: {rmse:.3f}, R²: {r2:.3f}")
    return rmse, r2

In [ ]:
lnr_rmse, lnr_r2 = evaluate(y_test, y_pred_lnr, "Linear Regression")
ridge_rmse, ridge_r2 = evaluate(y_test, y_pred_ridge, "Ridge Regression")
lasso_rmse, lasso_r2 = evaluate(y_test, y_pred_lasso, "Lasso Regression")

Linear Regression -> RMSE: 5.286, R²: 0.580
Ridge Regression -> RMSE: 5.280, R²: 0.581
Lasso Regression -> RMSE: 5.809, R²: 0.493


In [ ]:
from sklearn.linear_model import Lasso
lasso_tuned = Lasso(alpha=0.01)  # much smaller penalty
lasso_tuned.fit(X_train_scaled, y_train)

y_pred_l_tuned = lasso_tuned.predict(X_test_scaled)



In [ ]:
l_tuned_rmse, l_tuned_r2 = evaluate(y_test, y_pred_l_tuned, "Lasso (alpha=0.01)")

Lasso (alpha=0.01) -> RMSE: 5.280, R²: 0.581


In [ ]:
import joblib

joblib.dump(lasso_tuned, 'house_price_model.pkl')
joblib.dump(scale, 'scaler.pkl')

['scaler.pkl']